# Tập 4.2 · Đường tắt hay đường vòng? MDP 🎲

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap42/tap42_mdp.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 4 · ⏱ ~35 phút · Giáo trình: Principles of Robot Autonomy, [Chương 17](https://porabook.com/book/decision-making/)

Đường tắt sát cầu thang hay đường vòng an toàn, khi sàn trơn? Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video.
2. **Tự viết** xác suất chuyển, đánh giá chính sách và mô phỏng Monte Carlo (3 bài), có ô chấm tự động.
3. Vẽ kho hàng và chi phí kỳ vọng theo độ trơn.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap42")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import mdp as goc                  # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
for eps in (0.0, 0.1, 0.2):
    Jt, Ja = goc.danh_gia(goc.pi_tat, eps)[goc.S], goc.danh_gia(goc.pi_an_toan, eps)[goc.S]
    print(f"ε = {eps:.1f}: J đường tắt {Jt:6.2f} · J đường vòng {Ja:6.2f} · J* {goc.toi_uu(eps)[0][goc.S]:6.2f}")

## 2. Bài tập

$J_T(x) = g_T(x)$, $\quad J_t(x) = g(x) + \sum_{x'} p(x' \mid x, \pi(x))\,J_{t+1}(x')$; mỗi hướng có xác suất $(1-\varepsilon)[\text{đúng lệnh}] + \varepsilon/4$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `chuyen(x, u, eps)` | dict {ô mới: xác suất} |
| **2** | `danh_gia(pi, eps)` | dict {ô: $J_0$} |
| **3** | `mo_phong(pi, eps, n, seed)` | tỉ lệ tới G, tỉ lệ rơi |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np

from mdp import CAU_THANG, G, HUONG, O_TRONG, S, T, buoc, g, gT  # noqa: F401  (thế giới có sẵn, trong ../sim)


def chuyen(x, u, eps):
    """Bài 1 (Eq. 17.8): xác suất chuyển p(x_{t+1} | x_t = x, u_t = u), trả về dict {ô_mới: xác_suất}.

    Với mỗi hướng d trong HUONG: xác suất đi hướng đó = (1 − ε)·[d là hướng u] + ε/4; ô tới = buoc(x, vectơ của d).
    Nhiều hướng có thể dẫn tới cùng một ô (vd. đâm tường thì đứng yên): nhớ CỘNG DỒN xác suất."""
    raise NotImplementedError("TODO Bài 1: xác suất chuyển")


def danh_gia(pi, eps):
    """Bài 2 (Eq. 17.11): chi phí kỳ vọng J_π(x) của chính sách π (hàm x → lệnh), lùi từ cuối:

        J_T(x) = gT(x);   J_t(x) = g(x) + Σ_{x'} p(x' | x, π(x))·J_{t+1}(x')     (lặp T lần)
    Trả về dict {ô: J_0(ô)} cho mọi ô trong O_TRONG."""
    raise NotImplementedError("TODO Bài 2: đánh giá chính sách")


def mo_phong(pi, eps, n=2000, seed=42):
    """Bài 3 (Monte Carlo): cho robot chạy n lần, mỗi lần T bước từ S theo π, đếm tỉ lệ tới G và tỉ lệ rơi cầu thang.

    rng = np.random.default_rng(seed); hướng ngẫu nhiên lấy theo thứ tự list(HUONG.values()). Mỗi bước, gọi rng ĐÚNG thứ tự:
        nếu rng.random() < eps: hướng = list(HUONG.values())[rng.integers(4)]    (trượt)
        ngược lại:              hướng = HUONG[pi(x)]
        x = buoc(x, hướng)
    Trả về dict {"toi": tỉ lệ kết thúc ở G, "roi": tỉ lệ kết thúc ở cầu thang}."""
    raise NotImplementedError("TODO Bài 3: mô phỏng Monte Carlo")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong Bài 2, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Tìm độ trơn ε mà hai đường có chi phí bằng nhau (chia đôi).

In [ ]:
lo, hi = 0.0, 0.1
for _ in range(30):
    mid = (lo + hi) / 2
    if goc.danh_gia(goc.pi_tat, mid)[goc.S] < goc.danh_gia(goc.pi_an_toan, mid)[goc.S]:
        lo = mid
    else:
        hi = mid
print(f"hoà vốn ở ε ≈ {lo:.4f}: J = {goc.danh_gia(goc.pi_tat, lo)[goc.S]:.2f}")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap42).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)